In [6]:
from sklearn.ensemble import GradientBoostingRegressor,GradientBoostingClassifier
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv("../Datasets/Sonar.csv")

In [3]:
X=df.drop(columns=["Class"])
encoder=LabelEncoder()
y=encoder.fit_transform(df["Class"])

In [4]:
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [7]:
learn_rate=np.linspace(0.01,0.8,20)
no_of_estimate=[50,100,200]
depths=[3,5,None]
scores=[]
for r in tqdm(learn_rate):
    for n in no_of_estimate:
        for d in depths:
            gbm=GradientBoostingClassifier(n_estimators=n,learning_rate=r,max_depth=d,random_state=25)
            gbm.fit(X_train,Y_train)
            y_pred=gbm.predict(X_test)
            y_pred_prob=gbm.predict_proba(X_test)
            scores.append([r,n,d,log_loss(Y_test,y_pred_prob)])

score_df=pd.DataFrame(scores,columns=["learning_rate","n_estimators","depth","log_loss"]).sort_values("log_loss",ascending=True)
print(score_df)

100%|██████████| 20/20 [00:27<00:00,  1.38s/it]

     learning_rate  n_estimators  depth  log_loss
27        0.134737            50    3.0  0.428835
18        0.093158            50    3.0  0.442830
12        0.051579           100    3.0  0.446742
9         0.051579            50    3.0  0.457463
45        0.217895            50    3.0  0.458317
..             ...           ...    ...       ...
176       0.800000           100    NaN  4.116502
173       0.800000            50    NaN  4.116502
136       0.633684            50    5.0  4.276456
142       0.633684           200    5.0  4.276456
139       0.633684           100    5.0  4.276456

[180 rows x 4 columns]


In [8]:
pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 5.2 MB/s eta 0:00:14
   -- ------------------------------------- 4.7/69.5 MB 17.0 MB/s eta 0:00:04
   ----- ---------------------------------- 9.7/69.5 MB 19.6 MB/s eta 0:00:04
   -------- ------------------------------- 14.7/69.5 MB 21.5 MB/s eta 0:00:03
   ---------- ----------------------------- 18.1/69.5 MB 20.8 MB/s eta 0:00:03
   -------------- ------------------------- 25.2/69.5 MB 22.9 MB/s eta 0:00:02
   -------------------- ------------------- 34.9/69.5 MB 26.9 MB/s eta 0:00:02
   ------------------------- -------------- 44.0/69.5 MB 29.3 MB/s eta 0:00:01
   ---------------------------- ----------- 50.3/69.5 MB 29.6 MB/s eta 0:00:01
   -------------------------------- ------- 56.9/69.5 MB 29.8 MB/s eta 0:00:01
   ------------------------------------ --- 63.2/69.5 MB 30.1 MB/s eta 0:00:01
   ---------------------------------------  69.5/69.5 MB 30.3 MB/

In [11]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [10]:
pip install lightgbm
pip install xgboost

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
learn_rate=np.linspace(0.01,0.8,20)
no_of_estimate=[50,100,200]
depths=[3,5,None]
scores=[]
for r in tqdm(learn_rate):
    for n in no_of_estimate:
        for d in depths:
            xg=XGBClassifier(n_estimators=n,learning_rate=r,max_depth=d,random_state=25)
            xg.fit(X_train,Y_train)
            y_pred=xg.predict(X_test)
            y_pred_prob=xg.predict_proba(X_test)
            scores.append([r,n,d,log_loss(Y_test,y_pred_prob)])

score_df=pd.DataFrame(scores,columns=["learning_rate","n_estimators","depth","log_loss"]).sort_values("log_loss",ascending=True)
print(score_df)

100%|██████████| 20/20 [00:11<00:00,  1.77it/s]

     learning_rate  n_estimators  depth  log_loss
65        0.301053            50    NaN  0.422442
64        0.301053            50    5.0  0.422442
145       0.675263            50    5.0  0.429255
146       0.675263            50    NaN  0.429255
119       0.550526            50    NaN  0.431690
..             ...           ...    ...       ...
1         0.010000            50    5.0  0.577834
2         0.010000            50    NaN  0.577834
0         0.010000            50    3.0  0.588441
105       0.467368           200    3.0  0.599991
102       0.467368           100    3.0  0.599992

[180 rows x 4 columns]


In [19]:
learn_rate = np.linspace(0.01, 0.8, 20)
no_of_estimate = [50, 100, 200]
depths = [3, 5, None]

scores = []


for r in tqdm(learn_rate, desc="Grid Search Progress"):
    for n in no_of_estimate:
        for d in depths:
            model = LGBMClassifier(
                n_estimators=n,
                learning_rate=r,
                max_depth=d,
                random_state=25,
                verbose=-1  # Suppress LightGBM output warnings
            )
            # Fit on training data
            model.fit(X_train, Y_train)

            # Predict probabilities on validation data (use X_val/Y_val to avoid test leakage)
            y_pred_prob = model.predict_proba(X_test)

            # Calculate log_loss
            loss = log_loss(Y_test, y_pred_prob)
            scores.append([r, n, d, loss])

# Convert to DataFrame and sort by lowest log_loss
score_df = pd.DataFrame(
    scores,
    columns=["learning_rate", "n_estimators", "depth", "log_loss"]
).sort_values("log_loss", ascending=True).reset_index(drop=True)

print(score_df.head())

Grid Search Progress: 100%|██████████| 20/20 [00:02<00:00,  8.52it/s]

   learning_rate  n_estimators  depth  log_loss
0       0.093158            50    NaN  0.390546
1       0.093158            50    5.0  0.390546
2       0.134737            50    NaN  0.393635
3       0.134737            50    5.0  0.393635
4       0.176316            50    5.0  0.398391


In [16]:
pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 4.8 MB/s eta 0:00:21
   - -------------------------------------- 4.2/100.2 MB 14.8 MB/s eta 0:00:07
   ----- ---------------------------------- 12.6/100.2 MB 27.7 MB/s eta 0:00:04
   ------ --------------------------------- 16.0/100.2 MB 23.8 MB/s eta 0:00:04
   ------- -------------------------------- 19.4/100.2 MB 22.0 MB/s eta 0:00:04
   -------- ------------------------------- 22.0/100.2 MB 20.7 MB/s eta 0:00:04
   -------------- ------------------------- 37.5/100.2 MB 29.4 MB/s eta 0:00:03
   ------------------- -------------------- 49.8/100.2 MB 33.6 MB/s eta 0:00:02
   ------------------------ --------------- 61.6/100.2 MB 36.6 MB/s eta 0:00:02
   ----------------------------- ---------- 73.9/100.2 MB 39.2 MB/s eta 0:00:01
   ---------------------------------- ----- 86.5/100.2 MB 41.5 MB/s eta 0:00:01
   ---------------------------------------  99.1/100

In [20]:
from catboost import CatBoostClassifier

In [22]:
learn_rate=np.linspace(0.01,0.8,20)
no_of_estimate=[50,100,200]
depths=[3,5,None]
scores=[]
for r in tqdm(learn_rate):
    for n in no_of_estimate:
        for d in depths:
            xg=CatBoostClassifier(n_estimators=n,learning_rate=r,max_depth=d,random_state=25,verbose=0)
            xg.fit(X_train,Y_train)
            y_pred=xg.predict(X_test)
            y_pred_prob=xg.predict_proba(X_test)
            scores.append([r,n,d,log_loss(Y_test,y_pred_prob)])

score_df=pd.DataFrame(scores,columns=["learning_rate","n_estimators","depth","log_loss"]).sort_values("log_loss",ascending=True)
print(score_df)

100%|██████████| 20/20 [00:36<00:00,  1.83s/it]

     learning_rate  n_estimators  depth  log_loss
21        0.093158           100    3.0  0.399040
13        0.051579           100    5.0  0.401615
22        0.093158           100    5.0  0.405820
17        0.051579           200    NaN  0.408290
16        0.051579           200    5.0  0.409212
..             ...           ...    ...       ...
115       0.508947           200    5.0  0.786656
165       0.758421           100    3.0  0.791089
177       0.800000           200    3.0  0.824517
174       0.800000           100    3.0  0.825518
171       0.800000            50    3.0  0.876743

[180 rows x 4 columns]
